# Using Saxon (XPath, XSLT, XQuery) in Python

An exploratory walkthrough.

## Installing and Using Saxon with Python

You can write your own functions in Python, but most of the time we import *libraries* (collections of pre-built functions) to handle complex tasks like data analysis or XML processing.

To use XML-related languages (XPath, XSLT, XQuery) within Python, we need to:
1. **Install** the Saxon *processor* (the engine that executes XSLT/XPath/XQuery code)
2. **Import** the Saxon Python *library* (the Python interface that lets us interact with the processor)

### Installation (one time only)

Install Saxon using pip on the command line:
```bash
pip install saxonche
```

This installs the open-source Saxon-HE (Home Edition) for Python. See the [official documentation](https://www.saxonica.com/saxon-c/index.xml) for more details.

### How Saxon Works with Python

Python uses Saxon as a processor for XSLT, XPath, and XQuery code. This means Saxon executes the transformations using its full XSLT 3.0 engine rather than reimplementing these languages in Python. This approach gives us access to the complete, standards-compliant implementations of these W3C specifications.

## Why Use Saxon with Python?

Python's most popular XML libraries—[lxml](https://lxml.de/) and [BeautifulSoup](https://beautiful-soup-4.readthedocs.io/en/latest/)—are excellent tools for basic XML parsing and HTML scraping, but they don't provide complete implementations of the W3C specifications for XPath, XSLT, and XQuery. [Saxon](https://www.saxonica.com/saxon-c/index.xml) gives you full standards-compliant access to XPath 3.1, XSLT 3.0, and XQuery 3.1, including features like `xsl:for-each-group`, regular expressions in paths, higher-order functions, and complex transformations that these other libraries simply can't handle. If you need the complete XML processing toolkit that digital humanities scholars typically use in oXygen or on the command line, Saxon is the Python solution.

### Importing the Library

Now that we have Saxon installed, let's import the library to access its functions.

**Note on file paths**: This notebook includes examples that read files from your local file system (like XSLT stylesheets). To ensure these examples work correctly, you should clone the course repository and run this notebook from within that directory. This way Python will be able to find files using their relative paths. If you're copying and pasting code snippets into a Python REPL or running them from a different directory, you'll need to either adjust the file paths or download the necessary files to your working directory.

Alternatively, you could work directly from the URL: 
```python
# Compile the stylesheet directly from a URL
stylesheet = xslt_proc.compile_stylesheet(
    stylesheet_file="http://dh.obdurodon.org/bad-hamlet.xsl"
)
```

Note: Saxon can read XSLT stylesheets directly from HTTP URLs using the `stylesheet_file` parameter, just as it can read XML documents with `parse_xml(xml_uri=...)`.


In [2]:
from saxonche import PySaxonProcessor

Explanation of the import: `PySaxonProcessor` is the main "factory" class that creates the specialized processors you need (XSLT, XPath, XQuery, Schema validation). You use PySaxonProcessor to create these other processor objects, which then do the actual XML work. The library also includes classes representing XML data (nodes, atomic values, etc.), but `PySaxonProcessor` is where you always start.

## XPath

Let's run some basic XPath queries on our friend ["Bad Hamlet"](http://dh.obdurodon.org/bad-hamlet.xml). First, let's read the XML in as a local variable:

In [3]:
# Create the main Saxon processor
proc = PySaxonProcessor(license=False)

# Parse the XML and save it as a variable
bh_xml = proc.parse_xml(xml_uri="http://dh.obdurodon.org/bad-hamlet.xml")

# Print first 300 characters to verify it loaded
print(f"First 300 chars:\n{bh_xml.to_string()[:300]}")


First 300 chars:
<TEI xmlns="http://www.tei-c.org/ns/1.0" xml:id="sha-ham">
    <!-- Beware!! Munged version for teaching purposes!! -->
   <teiHeader>
      <fileDesc>
         <titleStmt>
            <title>Hamlet, Prince of Denmark</title>
            <author>William Shakespeare</author>
            <respStmt>
  


Now let's use XPath to pick up the first stage direction of each scene (`//div/div/stage[1]`). To do this, we need to set up the xpath processor, then set the XML document as the context item (the starting point for XPath evaluation), then run the query.

In [14]:
# Create XPath processor
xpath_proc = proc.new_xpath_processor()

# Declare the TEI namespace
xpath_proc.declare_namespace('tei', 'http://www.tei-c.org/ns/1.0')

# Set the document as the context item
xpath_proc.set_context(xdm_item=bh_xml)

# Tell XPath to match unprefixed names to any namespace
#xpath_proc.set_unprefixed_element_matching_policy(1)

# Use the namespace prefix in your query
result = xpath_proc.evaluate('//tei:div/tei:div/tei:stage[1]')

# Print results
for item in result:
        print(item.string_value)

 Elsinore. A platform before the castle. FRANCISCO at his post. 
A room of state in the castle.
A room in Polonius' house.
The platform.
Another part of the platform.
A room in Polonius' house.
A room in the castle.
A room in the castle.
A hall in the castle.
A room in the castle.
The Queen's closet.
A room in the castle.
Another room in the castle.
Another room in the castle.
A plain in Denmark.
Elsinore. A room in the castle.
Another room in the castle.
Another room in the castle.
A churchyard.
A hall in the castle.


This code just prints out the results within Python: big deal. If we are bothering to implement XPath from a Python environment, it is probably because we want to get the resulting data into some kind of Pythonic format so that we can do something else with it.

Let's start with something simple like capturing the resultant XPath string values associated with the nodes in a [Python list](https://www.w3schools.com/python/python_lists.asp). 

Before we extract the string values, let's see what XPath actually returns. The result contains node objects that preserve the XML structure:

In [15]:
# First, let's look at what the XPath result actually contains
print("Example node object:")
print(result[0])
print("\nString value of that same node:")
print(result[0].string_value)

# Create an empty list to store the results
stage_directions = []

# Loop through the XPath results and append string values to the list
for item in result:
    stage_directions.append(item.string_value)

# Display the list
print(stage_directions)

Example node object:
<stage xmlns="http://www.tei-c.org/ns/1.0"> Elsinore. A platform before the castle. FRANCISCO at his post. </stage>

String value of that same node:
 Elsinore. A platform before the castle. FRANCISCO at his post. 
[' Elsinore. A platform before the castle. FRANCISCO at his post. ', 'A room of state in the castle.', "A room in Polonius' house.", 'The platform.', 'Another part of the platform.', "A room in Polonius' house.", 'A room in the castle.', 'A room in the castle.', 'A hall in the castle.', 'A room in the castle.', "The Queen's closet.", 'A room in the castle.', 'Another room in the castle.', 'Another room in the castle.', 'A plain in Denmark.', 'Elsinore. A room in the castle.', 'Another room in the castle.', 'Another room in the castle.', 'A churchyard.', 'A hall in the castle.']


This code saved a list of *strings*, which means we lost all of the nodal data — which we may or may not have wanted. We could also create a list of nodes (i.e., preserve the XML tree data), and then still have the option to extract the string values as needed:

In [6]:
# List of strings (loses node data)
stage_strings = [item.string_value for item in result]

# List of nodes (preserves node data)
stage_nodes = list(result)

# Extract string values from nodes when needed
print(stage_nodes[3].string_value)

The platform.


This is already potentially useful. For instance, I could create a nice little tool for navigating the text: how about a Python function that uses the XPath processor we created as the first argument; XPath as the second argument; and a regular expression search for the third? 

First we need to import the regular expression library (which we would ordinarily do at the top of our Python script along with the others).

In [7]:
import re #regular expressions library

In [8]:
def text_comber(proc, xml_doc, xpath_expr, regex_pattern):
    """Extract text from XML nodes that match a regex pattern"""
    # Run XPath query using the same processor that created the doc
    xpath_proc = proc.new_xpath_processor()
    xpath_proc.set_context(xdm_item=xml_doc)
    result = xpath_proc.evaluate(xpath_expr)
    
    # Filter with regex
    matches = []
    if result:
        for node in result:
            text = node.string_value
            if re.search(regex_pattern, text):
                matches.append(text)
    
    return matches


This defines a function, which does something different each time we run it depending on the arguments. 

In [9]:
# Example usage - pass both the processor and document
results = text_comber(
    proc,                # The processor we created
    bh_xml,              # The document created by that processor
    "//*:stage",        # XPath expression
    r"(platform)|(room)"        # Regex pattern to search for
)

print(f"Found {len(results)} matches:")
for match in results[:5]:
    print(f"  - {match}")

Found 15 matches:
  -  Elsinore. A platform before the castle. FRANCISCO at his post. 
  - A room of state in the castle.
  - A room in Polonius' house.
  - The platform.
  - Another part of the platform.


In [10]:
text_comber(proc, bh_xml, "//*:stage", r"castle")

[' Elsinore. A platform before the castle. FRANCISCO at his post. ',
 'A room of state in the castle.',
 'A room in the castle.',
 'A room in the castle.',
 'A hall in the castle.',
 'A room in the castle.',
 'A room in the castle.',
 'Another room in the castle.',
 'Another room in the castle.',
 'Elsinore. A room in the castle.',
 'Another room in the castle.',
 'Another room in the castle.',
 'A hall in the castle.']

So that's a nice little tool for navigating a text — but not that much better than what you might do just using an editor like oXygen. More likely the reason you might wish to work within the Python environment would be to take advantage of other methods and libraries.

- **Statistical Analysis with Pandas**: Use [pandas](https://pandas.pydata.org/) DataFrames to count and analyze patterns (e.g., which characters speak most often, average speech length by character)

- **Data Visualization with Matplotlib**: Create charts and graphs with [matplotlib](https://matplotlib.org/) to visualize patterns in your text (e.g., bar charts of character speech counts, line graphs showing dialogue distribution across acts)

- **Network Analysis**: Use [NetworkX](https://networkx.org/) to analyze relationships between characters based on co-occurrence or speaking order, revealing social structures in the text

- **Text Mining Across Multiple Documents**: Process multiple XML files in a loop to compare patterns across different texts or editions

- **Machine Learning / Topic Modeling**: Use [scikit-learn](https://scikit-learn.org/) to identify distinctive vocabulary or discover hidden themes through TF-IDF analysis or topic modeling

- **Exporting to Other Formats**: Convert your XPath results to CSV, JSON, or Excel formats for analysis in other tools like R, Excel, or Tableau

- **Combining with Web APIs**: Integrate your textual data with external services (e.g., geocode place names, look up biographical information, retrieve images) using the [requests](https://requests.readthedocs.io/) library


Let's experiment with combining XPath (for returning particular parts of the text) with "sentiment analysis" using [NLTK](https://www.nltk.org/) (Natural Language Toolkit, a popular Python linguistics library.)

[Sentiment analysis](https://www.datacamp.com/tutorial/text-analytics-beginners-nltk) is:

> ... a technique used to determine the emotional tone or sentiment expressed in a text. It involves analyzing the words and phrases used in the text to identify the underlying sentiment, whether it is positive, negative, or neutral.

As always, we have to first import the relevant library:

In [11]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# nltk.download('vader_lexicon'): need to run this once to download the lexicon


In [12]:
# Extract all speeches by Hamlet
speeches = text_comber(proc, bh_xml, "//*:sp[@who='Hamlet']/*:l", r".")

# Analyze sentiment of each speech
sia = SentimentIntensityAnalyzer()
for speech in speeches[:5]:
    sentiment = sia.polarity_scores(speech)
    print(f"Speech: {speech[:50]}...")
    print(f"Sentiment: {sentiment['compound']}\n")

Speech: A little more than kin, and less than
            ...
Sentiment: 0.4804

Speech: Not so, my lord; I am too much in the
            ...
Sentiment: 0.0

Speech: Ay, madam, it is common....
Sentiment: 0.0

Speech: Seems, madam! nay, it is; I know not
             ...
Sentiment: 0.0

Speech: 'Tis not alone my inky cloak, good
               ...
Sentiment: 0.5632



That... worked! I don't really understand how these numbers are calculated, but positive numbers (0 to +1) = positive sentiment and negative numbers (-1 to 0) = negative sentiment. The point is that we successfully combined NLTK methods and XPath methods within Python to do something (potentially) useful.

## XSLT

We can also read in existing XSLT stylesheets and use them to transform XMl. Essentially, we are doing the same thing that oXygen does within the debugger, but within the Python environment.

In [13]:
# Create XSLT processor
xslt_proc = proc.new_xslt30_processor()

# Compile the stylesheet
stylesheet = xslt_proc.compile_stylesheet(stylesheet_file="bad-hamlet.xsl")

# Transform the document and save as variable
output = stylesheet.transform_to_string(xdm_node=bh_xml)

# Check the output
print(output[:500])  # First 500 characters

<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE html
  SYSTEM "about:legacy-compat">
<html xmlns="http://www.w3.org/1999/xhtml">
   <head>
      <title>Hamlet</title>
      <style type="text/css">
                    
                    .lg {
                        display: block;
                        margin-left: 1em;
                    }</style>
   </head>
   <body>
      <h1>Hamlet</h1>
      <section>
         <h2>Act 1</h2>
         <section>
            <h3>Act 1, Scene 1</h3>
     


Once again, so far we've only replicated what oXygen already does just fine in the debugger. But being able to do this within Python opens up options. For instance, we could iterate over a directory of XML documents, using the same transformation on each of them (which we could also do on command line, or with the `collection()` function). Or we could use the output file (whether html, XML in another form, SVG, CSV, or something else) to interact with other Python libraries.

For instance, imagine a project in which you are charting Marco Polo's journey and have marked up the text with a unique identifier for each stop along the way, like `<placeName ref="PLACE_042">Cathay</placeName>`. You could extract those elements using XPath, use the associated unique identifier attribute (`ref="PLACE_042"`) to pull the latitude longitude coordinates from a tabular data using [Pandas](https://pandas.pydata.org/):

| id | place_name | latitude | longitude |
|----|------------|----------|-----------|
| PLACE_042 | Cathay | 39.9042 | 116.4074 |
| PLACE_087 | Hormuz | 27.1865 | 56.2721 |
| PLACE_103 | Samarkand | 39.6270 | 66.9750 |

Then you could use the [Geopandas library](https://geopandas.org/en/stable/) to plot these coordinates on a map — all without ever leaving the Python environment.